In [ ]:
!pip install -q qqdm

In [ ]:
!gdown --id '1BazldgZLF-OH5kp5SXRKFUn3kn3ITVWx' --output data-bin.tar.xz

In [ ]:
!tar Jxvf data-bin.tar.xz
!rm data-bin.tar.xz

## 2. Import packages

In [ ]:
import numpy as np
import random

import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
import torchvision.transforms as transforms

from torch import nn
import torch.nn.functional as F
from torch.autograd import Variable
import torchvision.models as models
from torch.optim import Adam, AdamW

from sklearn.cluster import MiniBatchKMeans
from scipy.cluster.vq import vq, kmeans

from qqdm import qqdm, format_str
import pandas as pd


def same_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


same_seeds(2000)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Loading data

In [ ]:
train = np.load("data-bin/trainingset.npy", allow_pickle=True)
test = np.load("data-bin/testingset.npy", allow_pickle=True)

print(train.shape)
print(test.shape)

## 4. Fully-Connected Autoencoder (fcn_autoencoder)

- A baseline MLP-based autoencoder that flattens 2D images into 1D vectors for compression and reconstruction.


In [ ]:
class fcn_autoencoder(nn.Module):
    def __init__(self):
        super(fcn_autoencoder, self).__init__()
        # Compress 2D flattened image into lower dimensions
        self.encoder = nn.Sequential(
            nn.Linear(64 * 64 * 3, 128),
            nn.ReLU(True),
            nn.Linear(128, 64),
            nn.ReLU(True),
            nn.Linear(64, 12),
            nn.ReLU(True),
            nn.Linear(12, 3),
        )

        # Reconstruct lower dimensional latent variables back to original size
        self.decoder = nn.Sequential(
            nn.Linear(3, 12),
            nn.ReLU(True),
            nn.Linear(12, 64),
            nn.ReLU(True),
            nn.Linear(64, 128),
            nn.ReLU(True),
            nn.Linear(128, 64 * 64 * 3),
            # Bound output values to [-1, 1] to match normalized dataset range
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

## 5. Convolutional Autoencoder (conv_autoencoder)

- A CNN-based autoencoder that uses spatial convolutions and batch normalization layers to stabilize training and retain image structure.


In [ ]:
class conv_autoencoder(nn.Module):
    def __init__(self):
        super(conv_autoencoder, self).__init__()
        # Encoder: extract spatial features using convolution layers
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 12, 4, stride=2, padding=1),
            nn.BatchNorm2d(12),  # Added to stabilize input distribution
            nn.ReLU(),
            nn.Conv2d(12, 24, 4, stride=2, padding=1),
            nn.BatchNorm2d(24),  # Added to stabilize input distribution
            nn.ReLU(),
            nn.Conv2d(24, 48, 4, stride=2, padding=1),
            nn.BatchNorm2d(48),  # Added to stabilize input distribution
            nn.ReLU(),
            # nn.Conv2d(48, 96, 4, stride=2, padding=1),  # Removed to reduce model size
            # nn.ReLU(),
        )

        # Decoder: reconstruct texture features using transposed convolutions
        self.decoder = nn.Sequential(
            # nn.ConvTranspose2d(96, 48, 4, stride=2, padding=1), # Removed to match encoder changes
            # nn.BatchNorm2d(48),
            # nn.ReLU(),
            nn.ConvTranspose2d(48, 24, 4, stride=2, padding=1),
            nn.BatchNorm2d(24),  # Added to stabilize input distribution
            nn.ReLU(),
            nn.ConvTranspose2d(24, 12, 4, stride=2, padding=1),
            nn.BatchNorm2d(12),  # Added to stabilize input distribution
            nn.ReLU(),
            nn.ConvTranspose2d(12, 3, 4, stride=2, padding=1),
            # Bound output values to [-1, 1] to align with data normalization
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

## 6. Variational Autoencoder (VAE) & Loss Function

- Implements a VAE using a reparameterization trick to model a continuous latent space distribution, optimized via a custom MSE and KLD joint loss function.


In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 12, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(12, 24, 4, stride=2, padding=1),
            nn.ReLU(),
        )

        # Map features to distribution parameters: mean (mu) and log variance (logvar)
        # self.enc_out_1 = nn.Sequential(
        #     nn.Conv2d(24, 48, 4, stride=2, padding=1),
        #     nn.ReLU(),)
        # self.enc_out_2 = nn.Sequential(
        #     nn.Conv2d(24, 48, 4, stride=2, padding=1),
        #     nn.ReLU(),)
        self.enc_out_1 = nn.Conv2d(24, 48, 4, stride=2, padding=1)
        self.enc_out_2 = nn.Conv2d(24, 48, 4, stride=2, padding=1)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(48, 24, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(24, 12, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(12, 3, 4, stride=2, padding=1),
            nn.Tanh(),
        )

    def encode(self, x):
        h1 = self.encoder(x)
        return self.enc_out_1(h1), self.enc_out_2(h1)

    def reparametrize(self, mu, logvar):
        # Reparameterization Trick: move stochasticity to input to keep gradients differentiable
        std = logvar.mul(0.5).exp_()
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


# def loss_vae(recon_x, x, mu, logvar, criterion):
#     mse = criterion(recon_x, x)  # mse loss
#     kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
#     return mse + kld


def loss_vae(recon_x, x, mu, logvar):
    # Reconstruction Loss: calculate sum of pixels per sample, then average over the batch
    mse_per_sample = F.mse_loss(recon_x, x, reduction="none").sum(dim=[1, 2, 3])
    mse = mse_per_sample.mean()

    # KLD Loss: constrain the latent space to fit a standard normal distribution N(0, I)
    kld_per_sample = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp(), dim=[1, 2, 3]
    )
    kld = kld_per_sample.mean()

    return mse + kld

## 7. ResNet Autoencoder

- Uses a pretrained ResNet-18 model (with the last FC layer removed) as a strong feature extractor, mapping data into embedding spaces, and recovers images via transposed convolutions and bilinear interpolation.


In [ ]:
class Resnet(nn.Module):
    def __init__(self, fc_hidden1=1024, fc_hidden2=768, drop_p=0.3, CNN_embed_dim=256):
        super(Resnet, self).__init__()

        self.fc_hidden1, self.fc_hidden2, self.CNN_embed_dim = (
            fc_hidden1,
            fc_hidden2,
            CNN_embed_dim,
        )

        # CNN architectures configuration
        self.ch1, self.ch2, self.ch3, self.ch4 = 16, 32, 64, 128
        self.k1, self.k2, self.k3, self.k4 = (
            (5, 5),
            (3, 3),
            (3, 3),
            (3, 3),
        )  # 2d kernal size
        self.s1, self.s2, self.s3, self.s4 = (
            (2, 2),
            (2, 2),
            (2, 2),
            (2, 2),
        )  # 2d strides
        self.pd1, self.pd2, self.pd3, self.pd4 = (
            (0, 0),
            (0, 0),
            (0, 0),
            (0, 0),
        )  # 2d padding

        # Encoding components
        resnet = models.resnet18(pretrained=False)
        modules = list(resnet.children())[:-1]  # delete the last fc layer.
        self.resnet = nn.Sequential(*modules)
        self.fc1 = nn.Linear(resnet.fc.in_features, self.fc_hidden1)
        self.bn1 = nn.BatchNorm1d(self.fc_hidden1, momentum=0.01)
        self.fc2 = nn.Linear(self.fc_hidden1, self.fc_hidden2)
        self.bn2 = nn.BatchNorm1d(self.fc_hidden2, momentum=0.01)

        self.fc3_mu = nn.Linear(
            self.fc_hidden2, self.CNN_embed_dim
        )  # output = CNN embedding latent variables

        # Sampling vector
        self.fc4 = nn.Linear(self.CNN_embed_dim, self.fc_hidden2)
        self.fc_bn4 = nn.BatchNorm1d(self.fc_hidden2)
        self.fc5 = nn.Linear(self.fc_hidden2, 64 * 4 * 4)
        self.fc_bn5 = nn.BatchNorm1d(64 * 4 * 4)
        self.relu = nn.ReLU(inplace=True)

        # Decoder
        self.convTrans6 = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels=64,
                out_channels=32,
                kernel_size=self.k4,
                stride=self.s4,
                padding=self.pd4,
            ),
            nn.BatchNorm2d(32, momentum=0.01),
            nn.ReLU(inplace=True),
        )
        self.convTrans7 = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels=32,
                out_channels=8,
                kernel_size=self.k3,
                stride=self.s3,
                padding=self.pd3,
            ),
            nn.BatchNorm2d(8, momentum=0.01),
            nn.ReLU(inplace=True),
        )

        self.convTrans8 = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels=8,
                out_channels=3,
                kernel_size=self.k2,
                stride=self.s2,
                padding=self.pd2,
            ),
            nn.BatchNorm2d(3, momentum=0.01),
            nn.Tanh(),  # y = (y1, y2, y3) \in [0 ,1]^3
        )

    def encode(self, x):
        x = self.resnet(x)  # Feature extraction via ResNet backbone
        x = x.view(x.size(0), -1)  # Flatten convolutional layer outputs

        # Forward through fully-connected layers
        if x.shape[0] > 1:
            x = self.bn1(self.fc1(x))
        else:
            x = self.fc1(x)
        x = self.relu(x)
        if x.shape[0] > 1:
            x = self.bn2(self.fc2(x))
        else:
            x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3_mu(x)
        return x

    def decode(self, z):
        if z.shape[0] > 1:
            x = self.relu(self.fc_bn4(self.fc4(z)))
            x = self.relu(self.fc_bn5(self.fc5(x))).view(-1, 64, 4, 4)
        else:
            x = self.relu(self.fc4(z))
            x = self.relu(self.fc5(x)).view(-1, 64, 4, 4)
        x = self.convTrans6(x)
        x = self.convTrans7(x)
        x = self.convTrans8(x)
        # Use bilinear interpolation to reconstruct final texture dimension to 64x64
        x = F.interpolate(x, size=(64, 64), mode="bilinear", align_corners=True)
        return x

    def forward(self, x):
        z = self.encode(x)
        x_reconst = self.decode(z)

        return x_reconst

## 8. Custom Dataset Class

- Custom Dataset module that converts image channel dimensions from HWC to PyTorch-standard CHW format, and scales values from integers [0, 255] to floats [-1.0, 1.0].


In [ ]:
class CustomTensorDataset(TensorDataset):
    """TensorDataset with support of transforms."""

    def __init__(self, tensors):
        self.tensors = tensors
        # Check and permute dimensions from standard HWC to PyTorch CHW format
        if tensors.shape[-1] == 3:
            self.tensors = tensors.permute(0, 3, 1, 2)
        self.transform = transforms.Compose(
            [
                transforms.Lambda(lambda x: x.to(torch.float32)),
                # Dynamic mapping to scale integer pixels [0, 255] into [-1.0, 1.0] floats
                transforms.Lambda(lambda x: 2.0 * x / 255.0 - 1.0),
                # transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
            ]
        )

    def __getitem__(self, index):
        x = self.tensors[index]
        if self.transform:
            # Mapping images to [-1.0, 1.0]
            x = self.transform(x)
        return x

    def __len__(self):
        return len(self.tensors)

## 9. Training Loop

- Initializes dataset pipelines using a random batch loader, matches the specified target architecture, runs training iterations over 100 epochs, and automatically saves the best-performing and final model parameters.


In [ ]:
num_epochs = 100
batch_size = 128
learning_rate = 1e-4

# Build training dataloader with RandomSampler (contains only normal images)
x = torch.from_numpy(train)
train_dataset = CustomTensorDataset(x)

train_sampler = RandomSampler(train_dataset)
train_dataloader = DataLoader(
    train_dataset, sampler=train_sampler, batch_size=batch_size
)

# Model and Optimizer Selection
model_type = "cnn"
model_classes = {
    "resnet": Resnet(),
    "fcn": fcn_autoencoder(),
    "cnn": conv_autoencoder(),
    "vae": VAE(),
}
model = model_classes[model_type].to(device)

# Loss and optimizer setup
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
best_loss = np.inf
model.train()

for epoch in range(num_epochs):
    tot_loss = []
    for data in train_dataloader:
        img = data.float().to(device)
        # Flatten image to 1D vector if using fully-connected structure
        if model_type == "fcn":
            img = img.view(img.shape[0], -1)

        output = model(img)
        # Apply specific VAE loss function branching
        if model_type == "vae":
            loss = loss_vae(output[0], img, output[1], output[2], criterion)
        else:
            loss = criterion(output, img)

        tot_loss.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    mean_loss = np.mean(tot_loss)
    # Check and dynamically save the historical optimal model parameters
    if mean_loss < best_loss:
        best_loss = mean_loss
        torch.save(model.state_dict(), f"best_model_{model_type}.pt")

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {mean_loss:.4f}")

# Save the final model weights after completing all epochs
torch.save(model.state_dict(), f"last_model_{model_type}.pt")

## 10. Inference & Evaluation (Version A & B)

- Loads the trained evaluation weights under memory-isolated modes, sequentially reconstructs individual testing inputs, isolates individual target components, aggregates total error counts into a standard square-root metric (RMSE), and logs predictions into Kaggle format.


In [ ]:
eval_batch_size = 200
# Build sequential testing dataloader
data = torch.tensor(test, dtype=torch.float32)
test_dataset = CustomTensorDataset(data)
test_sampler = SequentialSampler(test_dataset)
test_dataloader = DataLoader(
    test_dataset, sampler=test_sampler, batch_size=eval_batch_size
)
# Set reduction="none" to isolate pixel-wise sample errors independently
eval_loss = nn.MSELoss(reduction="none")
# Load matched structural parameters safely
checkpoint_path = f"last_model_{model_type}.pt"
model = model_classes[model_type].to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
out_file = "PREDICTION_FILE.csv"
anomality = []
# Version A Inference Logic
with torch.no_grad():
    for data in test_dataloader:
        img = data.float().to(device)
        if model_type == "fcn":
            img = img.view(img.shape[0], -1)

        output = model(img)
        if model_type == "vae":
            output = output[0]

        if model_type == "fcn":
            loss = eval_loss(output, img).sum(-1)
        else:
            loss = eval_loss(output, img).sum([1, 2, 3])
        anomality.append(loss)

anomality = torch.cat(anomality, dim=0)
# Transform pixel deviations into an RMSE standard anomaly score matrix
anomality = torch.sqrt(anomality).reshape(len(test), 1).cpu().numpy()

df = pd.DataFrame(anomality, columns=["Predicted"])
df.to_csv(out_file, index_label="Id")

In [ ]:
# Version B Inference Logic (Alternative Multi-Branch)
anomality = list()
with torch.no_grad():
    for i, data in enumerate(test_dataloader):
        if model_type in ["cnn", "vae", "resnet"]:
            img = data.float().to(device)
        elif model_type in ["fcn"]:
            img = data.float().to(device)
            img = img.view(img.shape[0], -1)
        else:
            img = data[0].to(device)
        output = model(img)
        if model_type in ["cnn", "resnet", "fcn"]:
            output = output
        elif model_type in ["res_vae"]:
            output = output[0]
        elif model_type in ["vae"]:
            output = output[0]
        if model_type in ["fcn"]:
            loss = eval_loss(output, img).sum(-1)
        else:
            loss = eval_loss(output, img).sum([1, 2, 3])
        anomality.append(loss)
anomality = torch.cat(anomality, axis=0)
# Transform sum of squares to standard RMSE values (Higher score means higher anomaly probability)
anomality = torch.sqrt(anomality).reshape(len(test), 1).cpu().numpy()

df = pd.DataFrame(anomality, columns=["Predicted"])
df.to_csv(out_file, index_label="Id")